# 03 · SInAS × GloBI — backbone matching, taxonomy and interaction network

**Data stream:** species, taxonomy and species interactions.
**Output feeds:** the species / taxonomy graph of the model, and notebook
04 (GBIF presence and biotic opportunity).

---

## What this notebook does

1. Matches every **SInAS** species name and every **GloBI** interaction
   partner name to **one taxonomy**, the GBIF Backbone, so that the same
   species gets the same ID in both datasets.
2. Builds a **taxonomy table** (species → genus → family → order → class →
   phylum → kingdom) for every taxon in the graph, including GloBI partners
   that are not in SInAS.
3. Classifies every GloBI interaction type into an ecological **category**
   and a **valence** for each side (is the partner good or bad for me?).
4. Extracts the interaction **network** around the SInAS species:
   * species ↔ species edges, and
   * **higher-order interactions (HOI)**: edges where one side is only known
     at genus / family / order / class level (e.g. "*Bombus terrestris*
     visits flowers of *Salix*"). These are kept when the higher taxon
     contains SInAS species or the other side is a SInAS species.

## Pipeline at a glance

| Step | What happens | Main output |
|---|---|---|
| 0 | Configuration, sources, helper functions | — |
| 1 | Read SInAS, clean names, parse establishment status | `sinas` |
| 2 | Match SInAS names to the GBIF Backbone | `sinas_res` |
| 3 | Scan GloBI (pass 1): collect every partner name | `globi_names` |
| 4 | Match GloBI names (+ fallback on GloBI's own GBIF IDs) | `globi_res` |
| 5 | Interaction ontology: category + valence per raw term | `ontology` |
| 6 | Scan GloBI (pass 2): extract species and HOI edges | `edges` |
| 7 | Taxonomy backbone table | `taxonomy` |
| 8 | Assemble the SInAS output with match and interaction flags | `sinas_out` |
| 9 | QA, incl. comparison with the previous version of the outputs | `qa` |
| 10 | Write outputs | files in `data/` |

## Outputs (written to `data/`)

| File | Content |
|---|---|
| `sinas_matched_species.csv` | All SInAS rows whose name matched a **species** in the backbone, with `gbif_*` keys, `is_alien` / `is_native` and interaction flags |
| `taxonomy_backbone.csv` | One row per taxon in the graph: rank, name and all parent keys / names |
| `matched_globi_network.csv` | Species ↔ species edges with category, valence and evidence counts |
| `matched_globi_network_higher_order.csv` | HOI edges (≥ 1 side above species level) |
| `interaction_type_ontology.csv` | Every raw GloBI interaction term → category, model edge type, valences, number of records |
| `name_matching_audit.csv` | Every input name with its match result and flags (for checking) |
| `sinas_globi_qa.json` | QA report |

Column names of the first three files are kept compatible with the model
notebook (`gbif_id`, `source_gbif_id`, `interaction_type`, …); the new
columns are additions.

## Design rules (read before editing)

1. **One taxonomy: the GBIF Backbone**, via the v1 name-match API. GBIF's
   occurrence filters also default to the Backbone, so the IDs used here
   match the IDs used in notebook 04. (GBIF's newer primary taxonomy is
   Catalogue of Life XR; switching would require changing both notebooks
   together.)
2. **Species-level key = `speciesKey`.** Subspecies, varieties and forms are
   rolled up to their species, and synonyms resolve to the accepted
   species, so *Acer negundo* var. *californicum* and *Negundo aceroides*
   both become *Acer negundo*.
3. **No false species.** A SInAS name GBIF can only match to a genus or
   higher (`HIGHERRANK`), or a fuzzy match below `MIN_FUZZY_CONFIDENCE`, is
   **not** used as a species. Those rows are listed in the audit file.
   (Previously such rows became "species" with a genus or even kingdom ID.)
4. **All matched SInAS species are kept**, also those without any GloBI
   interaction (`has_globi_interactions = False`). They still carry
   climate, trade and taxonomy signal. The model can filter on the flag.
5. **Valence is defined per raw interaction term**, for each side
   separately (e.g. `vectorOf` is good for the pathogen, neutral for the
   vector). Generic terms (`interactsWith`, `coOccursWith`,
   `associatedWith`) are **neutral** by default; set
   `GENERIC_ASSOCIATION_AS = "mutualism"` to reproduce the old behaviour.

## How to run

1. Install (once): `pip install pandas numpy aiohttp nest_asyncio tqdm`
2. Put `SInAS_3.1.1.csv` and `interactions.tsv.gz` in `../data/` (Step 0).
3. Set `CONTACT_EMAIL` in the configuration cell (GBIF asks API users to
   identify themselves).
4. *Run All*. The first run matches ~1 million names (≈ 20–30 min) and
   reads the 20-million-row GloBI file twice (≈ 10 min per pass). All GBIF
   answers are cached in `data/cache/`, so re-runs only query new names.

A description of every output column and its provenance is in
`sinas_globi_metadata.md`.

## Step 0 · Data sources and configuration

| File | Source | How to obtain |
|---|---|---|
| `SInAS_3.1.1.csv` | SInAS v3.1.1 — Gómez-Suárez, Laeseke & Seebens (2025), *Scientific Data* 12: 1914, [doi:10.1038/s41597-025-06379-6](https://doi.org/10.1038/s41597-025-06379-6) | **Manual download** from Zenodo: <https://zenodo.org/records/17727120> |
| `interactions.tsv.gz` | Global Biotic Interactions (GloBI), interpreted data product — Poelen, Simons & Mungall (2014), *Ecological Informatics* 24: 148–159, [doi:10.1016/j.ecoinf.2014.08.005](https://doi.org/10.1016/j.ecoinf.2014.08.005) | **Manual download**: Zenodo record <https://zenodo.org/records/3950590> → click *"latest version"* → file `interactions.tsv.gz` (~0.5–1 GB). Overview: <https://www.globalbioticinteractions.org/data>. Note the download date in the metadata file. |
| GBIF Backbone Taxonomy | GBIF Secretariat, GBIF Backbone Taxonomy, [doi:10.15468/39omei](https://doi.org/10.15468/39omei) | **Automatic**, via the name-match API `https://api.gbif.org/v1/species/match` |

> GloBI asks users to also credit the original data providers; the
> `reference` column of the network files keeps one citation per edge, and
> the full list is in the GloBI download (`citations.tsv.gz`).

In [20]:
import asyncio
import csv
import json
import random
import re
import warnings
import os
from collections import Counter
from pathlib import Path
from dotenv import load_dotenv

import aiohttp
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 40)

### Configuration

Paths and every tunable choice live here. Normally you only change `DATA`
and `CONTACT_EMAIL`.

In [ ]:
# ---------------------------------------------------------------- paths
load_dotenv(Path.cwd().parent / ".env")

DATA = Path.cwd().parent / "data"
SINAS_RAW_CSV = DATA / "SInAS_3.1.1.csv"
GLOBI_TSV = DATA / "interactions.tsv.gz"
CACHE_DIR = DATA / "cache"
NAME_CACHE = CACHE_DIR / "gbif_name_matches.jsonl"
ID_CACHE = CACHE_DIR / "gbif_usage_lookups.jsonl"

# ---------------------------------------------------------------- outputs
SINAS_OUT = DATA / "sinas_matched_species.csv"
TAXONOMY_OUT = DATA / "taxonomy_backbone.csv"
NET_SPECIES_OUT = DATA / "matched_globi_network.csv"
NET_HIGHER_OUT = DATA / "matched_globi_network_higher_order.csv"
ONTOLOGY_OUT = DATA / "interaction_type_ontology.csv"
AUDIT_OUT = DATA / "name_matching_audit.csv"
QA_OUT = DATA / "sinas_globi_qa.json"
BACKUP_PREVIOUS = True  # keep the old outputs as *_previous.csv to compare

# ---------------------------------------------------------------- GBIF API
GBIF_MATCH_URL = "https://api.gbif.org/v1/species/match"
GBIF_USAGE_URL = "https://api.gbif.org/v1/species/{}"
CONTACT_EMAIL = os.getenv("GBIF_EMAIL")  # <- set this
MAX_CONCURRENT = 50  # parallel requests (old notebook: 50 worked fine)
REQUEST_TIMEOUT = 30  # seconds
MAX_RETRIES = 4
MATCH_BATCH = 5000  # names per batch; the cache is written after each

# ---------------------------------------------------------------- matching
MIN_FUZZY_CONFIDENCE = 90  # fuzzy matches below this are rejected
SPECIES_RANKS = {"SPECIES", "SUBSPECIES", "VARIETY", "FORM",
                 "INFRASPECIFIC_NAME", "INFRASUBSPECIFIC_NAME",
                 "SUBVARIETY", "SUBFORM"}
# Ranks allowed as higher-order interaction endpoints. Phylum / kingdom
# partners ("eats Plantae") are present everywhere and carry no signal.
HOI_RANKS = ("GENUS", "FAMILY", "ORDER", "CLASS")
# A GloBI name that only matches at a higher rank (e.g. "Quercus sp.")
# is used as a higher-order endpoint; SInAS names never are.
ACCEPT_HIGHERRANK_FOR_GLOBI = True

# ---------------------------------------------------------------- network
# Keep HOI edges whose higher-taxon endpoint is one of these ancestors of
# a SInAS species (or whose other endpoint is a SInAS species).
HOI_ANCESTOR_LEVELS = ("genus", "family", "order", "class")
# Generic terms (interactsWith, coOccursWith, associatedWith): "neutral"
# (default, no valence) or "mutualism" (old behaviour).
GENERIC_ASSOCIATION_AS = "neutral"
GLOBI_CHUNK = 1_000_000  # rows per chunk when streaming the GloBI file

CACHE_DIR.mkdir(parents=True, exist_ok=True)
for f in (SINAS_RAW_CSV, GLOBI_TSV):
    print(f"[{'OK ' if f.exists() else 'MISSING'}] {f}")

[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\SInAS_3.1.1.csv
[OK ] c:\Users\simon\Documents\GitHub\horizon-scanner\data\interactions.tsv.gz


### Helper functions

(a) file and name helpers, (b) GBIF matching with an on-disk cache,
(c) the interaction ontology. Nothing is computed until Step 1.

In [ ]:
# ======================================================================
# (a) Files and names
# ======================================================================

def read_table_auto(path, **kwargs):
    """Read a delimited text file whose delimiter/encoding is unknown.

    The delimiter is detected with :class:`csv.Sniffer` on the first 64 kB
    (comma, semicolon, tab, pipe or space; SInAS is written by R with
    space-separated, double-quoted fields). If sniffing fails, pandas'
    own detector is used. Rows with a wrong number of fields are skipped
    with a warning instead of stopping the read.

    Parameters
    ----------
    path : str or Path
    **kwargs
        Passed on to :func:`pandas.read_csv`.

    Returns
    -------
    pandas.DataFrame
    """
    path = Path(path)
    last_err = None
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, "r", encoding=enc, newline="") as fh:
                sample = fh.read(65536)
            sample = sample[:sample.rfind("\n") + 1] or sample
            try:
                sep = csv.Sniffer().sniff(sample,
                                          delimiters=",;\t| ").delimiter
            except csv.Error:
                sep = None
            print(f"   {path.name}: encoding={enc}, sep={sep!r}")
            common = dict(encoding=enc, quotechar='"', **kwargs)
            if sep is None:
                return pd.read_csv(path, sep=None, engine="python",
                                   **common)
            try:
                return pd.read_csv(path, sep=sep, low_memory=False,
                                   **common)
            except pd.errors.ParserError as err:
                print(f"   C parser failed ({err}); skipping bad rows")
                bad_rows = []

                def _skip(fields):
                    bad_rows.append(fields)
                    return None

                df = pd.read_csv(path, sep=sep, engine="python",
                                 on_bad_lines=_skip, **common)
                if bad_rows:
                    warnings.warn(f"{path.name}: skipped {len(bad_rows)} "
                                  "malformed rows")
                return df
        except UnicodeDecodeError as err:
            last_err = err
    raise ValueError(f"Could not decode {path}: {last_err}")



def run_async(coro):
    """Run a coroutine both in a plain script and inside Jupyter.

    Jupyter already runs an event loop, so ``asyncio.run`` would fail
    there; ``nest_asyncio`` lets us re-enter the running loop instead.
    If the cell is interrupted, the task (and its workers) is cancelled,
    so no requests keep running in the background.
    """
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)
    import nest_asyncio
    nest_asyncio.apply()
    task = loop.create_task(coro)
    try:
        return loop.run_until_complete(task)
    except BaseException:  # interrupt or error: stop the workers too
        task.cancel()
        try:
            loop.run_until_complete(task)
        except BaseException:
            pass
        raise


def clean_name(name):
    """Normalise a scientific-name string for matching.

    Only whitespace and stray quotes are cleaned; authorship and rank
    markers are left for GBIF's matcher, which understands them.
    """
    if not isinstance(name, str):
        return None
    out = re.sub(r"\s+", " ", name).strip().strip('"').strip()
    return out or None


def clean_kingdom(value):
    """Kingdom hint for the matcher (None if empty / not informative)."""
    if not isinstance(value, str):
        return None
    value = value.strip()
    return value if value and value.lower() not in ("nan", "none") \
        else None

In [5]:
# ======================================================================
# (b) GBIF Backbone matching with an on-disk cache
# ======================================================================
MATCH_FIELDS = [
    "usageKey", "acceptedUsageKey", "scientificName", "canonicalName",
    "rank", "status", "matchType", "confidence",
    "kingdom", "phylum", "class", "order", "family", "genus", "species",
    "kingdomKey", "phylumKey", "classKey", "orderKey", "familyKey",
    "genusKey", "speciesKey",
]
LEVELS = ["kingdom", "phylum", "class", "order", "family", "genus"]


def _cache_key(name, kingdom):
    return f"{name}|{kingdom or ''}"


def load_jsonl_cache(path):
    """Read a JSONL cache into {key: record}; half-written lines skipped."""
    cache = {}
    if path.exists():
        with open(path, encoding="utf-8") as fh:
            for line in fh:
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    continue
                cache[rec["_key"]] = rec
    return cache


async def _get_json(session, url, params, sem):
    """GET with retries. Returns (ok, json_or_None).

    ``ok`` is False only for transient failures (timeouts, 429, 5xx),
    which are *not* cached so that a re-run retries them.
    """
    for attempt in range(MAX_RETRIES):
        try:
            async with sem:
                async with session.get(
                    url, params=params,
                    timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT),
                ) as resp:
                    if resp.status == 429 or resp.status >= 500:
                        await asyncio.sleep(2 ** attempt + random.random())
                        continue
                    if resp.status != 200:
                        return True, None
                    return True, await resp.json(content_type=None)
        except (asyncio.TimeoutError, aiohttp.ClientError):
            await asyncio.sleep(2 ** attempt + random.random())
    return False, None


async def _fetch_batch(requests):
    """Run [(key, url, params)] concurrently -> [(key, ok, data)]."""
    sem = asyncio.Semaphore(MAX_CONCURRENT)
    headers = {"User-Agent": f"IAS-horizon-scanner ({CONTACT_EMAIL})"}
    connector = aiohttp.TCPConnector(limit=MAX_CONCURRENT)
    async with aiohttp.ClientSession(headers=headers,
                                     connector=connector) as session:
        async def one(key, url, params):
            ok, data = await _get_json(session, url, params, sem)
            return key, ok, data
        return await asyncio.gather(*(one(*r) for r in requests))


def cached_requests(requests, cache_path, to_record, desc):
    """Query uncached requests in batches and append them to the cache.

    Parameters
    ----------
    requests : list of (key, url, params)
    cache_path : Path
        JSONL cache file.
    to_record : callable(data) -> dict
        Turns the API answer (or None) into the record stored.
    desc : str
        Progress-bar label.

    Returns
    -------
    dict
        {key: record} for every request that has an answer.
    """
    cache = load_jsonl_cache(cache_path)
    todo = list({r[0]: r for r in requests if r[0] not in cache}
                .values())
    print(f"   {desc}: {len(requests):,} requests | "
          f"{len(requests) - len(todo):,} cached | {len(todo):,} to query")
    n_failed = 0
    with open(cache_path, "a", encoding="utf-8") as fh:
        for start in tqdm(range(0, len(todo), MATCH_BATCH), desc=desc,
                          unit="batch"):
            batch = todo[start:start + MATCH_BATCH]
            for key, ok, data in run_async(_fetch_batch(batch)):
                if not ok:
                    n_failed += 1
                    continue
                rec = to_record(data)
                rec["_key"] = key
                cache[key] = rec
                fh.write(json.dumps(rec) + "\n")
            fh.flush()
    if n_failed:
        warnings.warn(f"{desc}: {n_failed:,} requests failed (network); "
                      "re-run this cell to retry them")
    return cache


def match_names(pairs, desc):
    """Match (name, kingdom) pairs to the GBIF Backbone (cached).

    Returns a DataFrame with ``name``, ``kingdom_hint`` and MATCH_FIELDS.
    """
    pairs = list(dict.fromkeys(pairs))
    requests = []
    for name, kingdom in pairs:
        params = {"name": name, "strict": "false", "verbose": "false"}
        if kingdom:
            params["kingdom"] = kingdom
        requests.append((_cache_key(name, kingdom), GBIF_MATCH_URL,
                         params))
    cache = cached_requests(
        requests, NAME_CACHE,
        lambda d: {f: (d or {}).get(f) for f in MATCH_FIELDS}, desc)
    rows = []
    for name, kingdom in pairs:
        rec = cache.get(_cache_key(name, kingdom))
        if rec is not None:
            rows.append({"name": name, "kingdom_hint": kingdom,
                         **{f: rec.get(f) for f in MATCH_FIELDS}})
    out = pd.DataFrame(rows, columns=["name", "kingdom_hint"]
                       + MATCH_FIELDS)
    out["kingdom_hint"] = out["kingdom_hint"].map(clean_kingdom)
    return out


def lookup_usage_keys(keys, desc):
    """Fetch backbone records for GBIF usage keys (cached).

    Returns {key: record in MATCH_FIELDS layout, matchType="GLOBI_ID"}.
    """
    def to_record(d):
        d = d or {}
        rec = {f: d.get(f) for f in MATCH_FIELDS}
        rec.update(usageKey=d.get("key"),
                   acceptedUsageKey=d.get("acceptedKey"),
                   status=d.get("taxonomicStatus"),
                   matchType="GLOBI_ID" if d else "NONE",
                   confidence=100 if d else 0)
        return rec
    requests = [(str(k), GBIF_USAGE_URL.format(k), {}) for k in keys]
    cache = cached_requests(requests, ID_CACHE, to_record, desc)
    return {int(k): cache[str(k)] for k in keys if str(k) in cache}


def resolve_matches(df, allow_higherrank):
    """Turn raw match records into graph nodes.

    Adds ``species_key`` (species-level key, synonyms and infraspecific
    names rolled up), ``node_key`` / ``node_rank`` (the taxon the name
    stands for in the graph: a species, or a higher taxon for HOI) and
    ``match_flag``.

    Parameters
    ----------
    df : pandas.DataFrame
        Output of :func:`match_names`.
    allow_higherrank : bool
        Accept HIGHERRANK matches as higher-taxon nodes (GloBI only).
    """
    out = df.copy()
    mt = out["matchType"].fillna("NONE")
    conf = pd.to_numeric(out["confidence"], errors="coerce").fillna(0)
    rank = out["rank"].fillna("")
    good = (mt.isin(["EXACT", "GLOBI_ID"])
            | ((mt == "FUZZY") & (conf >= MIN_FUZZY_CONFIDENCE)))
    if allow_higherrank:
        good |= (mt == "HIGHERRANK")
    out["match_flag"] = np.select(
        [mt.eq("NONE"), mt.eq("HIGHERRANK"),
         mt.eq("FUZZY") & (conf < MIN_FUZZY_CONFIDENCE), mt.eq("FUZZY"),
         mt.eq("GLOBI_ID")],
        ["no_match", "higherrank", "fuzzy_low_confidence", "fuzzy",
         "globi_gbif_id"], default="exact")

    accepted = np.where(out["status"].eq("SYNONYM")
                        & out["acceptedUsageKey"].notna(),
                        out["acceptedUsageKey"], out["usageKey"])
    accepted = pd.to_numeric(pd.Series(accepted, index=out.index),
                             errors="coerce")
    sp_key = pd.to_numeric(out["speciesKey"], errors="coerce")
    is_species = good & rank.isin(SPECIES_RANKS) & mt.ne("HIGHERRANK")
    out["species_key"] = sp_key.where(is_species)
    # accepted SPECIES record without a speciesKey: use its own key
    fill = is_species & out["species_key"].isna() & rank.eq("SPECIES")
    out.loc[fill, "species_key"] = accepted[fill]

    is_higher = good & rank.isin(HOI_RANKS) & out["species_key"].isna()
    out["node_key"] = out["species_key"].where(out["species_key"].notna(),
                                               accepted.where(is_higher))
    out["node_rank"] = np.where(out["species_key"].notna(), "SPECIES",
                                np.where(is_higher, rank, None))
    for col in ("species_key", "node_key"):
        out[col] = out[col].astype("Int64")
    return out

In [12]:
# ======================================================================
# (c) Interaction ontology: raw GloBI term -> category and valence
# ======================================================================
# Valence is given separately for the SOURCE and the TARGET of a GloBI
# record: +1 = the other side is good for me (food, host, mutualist,
# habitat), -1 = bad for me (predator, parasite, pathogen, competitor),
# 0 = neutral / unknown. `gnn` is the edge type used by the model graph
# (resource / enemy / mutualism / competition / none).
def _terms(category, gnn, src, tgt, names):
    return {n: (category, gnn, src, tgt) for n in names.split()}


TERM_TABLE = {
    # source consumes / parasitises / infects the target
    **_terms("resource", "resource", +1, -1, """
        eats preysOn kills consumes parasiteOf parasitizes endoparasiteOf
        ectoparasiteOf parasitoidOf endoparasitoidOf ectoparasitoidOf
        ectoParasitoid hyperparasiteOf hyperparasitoidOf kleptoparasiteOf
        pathogenOf rootparasiteOf hemiparasiteOf hasHost
        acquiresNutrientsFrom bloodFeedsOn seedEaterOf woodBorerOf
        leafMinerOf gallMakerOf defoliatorOf rootFeederOf grazesOn
        browses laysEggsOn laysEggsIn"""),
    # source uses the target without harming it (dead matter, nectar)
    **_terms("resource_nonlethal", "resource", +1, 0, """
        scavenges saprophyteOf nectarFeedsOn pollenFeedsOn"""),
    # mirror image of "resource": the target eats / parasitises the source
    **_terms("enemy", "enemy", -1, +1, """
        eatenBy preyedUponBy preyedOnBy killedBy hasParasite
        hasEndoparasite hasEctoparasite hasParasitoid hasEndoparasitoid
        hasEctoparasitoid hasHyperparasite hasHyperparasitoid
        hasKleptoparasite hasPathogen hostOf providesNutrientsFor
        hasEggsLaidOnBy hasEggsLaidInBy parasitizedBy infectedBy
        parasitoidBy pathogenBy"""),
    # both sides benefit
    **_terms("mutualism", "mutualism", +1, +1, """
        pollinates pollinatedBy visitsFlowersOf flowersVisitedBy visitedBy visits
        mutualistOf symbiontOf hasSymbiont mycorrhizalWith
        dispersalVectorOf hasDispersalVector dispersesSeedsOf disperses"""),
    # source benefits (habitat, transport, shelter); target unaffected
    **_terms("commensal", "mutualism", +1, 0, """
        commensalistOf epiphyteOf inquilineOf phoreticOf guestOf livesOn
        livesInsideOf livesUnder livesNear inhabits hasHabitat hasRoost
        nestedIn sheltersWithin hasVector"""),
    # target benefits; source unaffected
    **_terms("commensal_host", "mutualism", 0, +1, """
        hasCommensalist hasEpiphyte hasInquiline hasPhoretic hasGuest
        createsHabitatFor providesShelterTo hostsNestedIn vectorOf"""),
    # both sides harmed
    **_terms("competition", "competition", -1, -1, "competesWith"),
    # source harms the target without benefit to itself
    **_terms("antagonism", "competition", 0, -1, """
        allelopathOf allelopathicTo inhibits displaces antagonistOf
        interferesWith"""),
    # no ecological direction
    **_terms("neutral", "none", 0, 0, """
        interactsWith coOccursWith adjacentTo ecologicallyRelatedTo
        relatedTo perceives perceivedBy coRoostsWith livesWith
        associatedWith"""),
}
if GENERIC_ASSOCIATION_AS == "mutualism":  # reproduce the old ontology
    TERM_TABLE.update(_terms("mutualism", "mutualism", +1, +1,
                             "coOccursWith associatedWith"))


def classify_term(term):
    """Category, model edge type and valences for one raw GloBI term.

    Returns
    -------
    tuple
        (category, gnn_category, source_valence, target_valence, rule)
        where ``rule`` is "table", "heuristic" or "unmatched".
    """
    if not isinstance(term, str) or not term.strip():
        return ("unknown", "none", 0, 0, "unmatched")
    t = term.strip()
    if t in TERM_TABLE:
        return (*TERM_TABLE[t], "table")
    low = t.lower()
    if "by" in low and any(w in low for w in ("eat", "prey", "parasit",
                                               "kill", "infect", "attack")):
        return ("enemy", "enemy", -1, +1, "heuristic")
    if any(w in low for w in ("eat", "prey", "consume", "feed", "graze",
                              "browse", "parasit", "pathogen")):
        return ("resource", "resource", +1, -1, "heuristic")
    if any(w in low for w in ("symbio", "mutual", "pollin")):
        return ("mutualism", "mutualism", +1, +1, "heuristic")
    if any(w in low for w in ("compet", "inhib", "displac", "antagon")):
        return ("competition", "competition", -1, -1, "heuristic")
    return ("unknown", "none", 0, 0, "unmatched")

## Step 1 · SInAS — names and establishment status

1. Read `SInAS_3.1.1.csv`; clean the `taxon` and `location` strings.
2. Parse `establishmentMeans` into `is_alien` / `is_native`. Note that
   *non-native* contains the word *native*; the regex checks for alien
   terms first so non-native records are not filed as native.
3. If SInAS has a `kingdom` column it is passed to the matcher as a hint
   (helps with names shared by a plant and an animal genus).

In [7]:
print("Step 1: SInAS")
sinas = read_table_auto(SINAS_RAW_CSV)
sinas["taxon_clean"] = sinas["taxon"].map(clean_name)
sinas["location"] = sinas["location"].astype(str).str.strip()
kingdom_col = next((c for c in sinas.columns if c.lower() == "kingdom"),
                   None)
sinas["kingdom_hint"] = (sinas[kingdom_col].map(clean_kingdom)
                         if kingdom_col else None)

means = (sinas.get("establishmentMeans", pd.Series("", index=sinas.index))
         .fillna("").astype(str).str.lower().str.strip())
sinas["is_alien"] = means.str.contains(
    r"introduced|invasive|naturali[sz]ed|alien|non[\s_-]?native")
sinas["is_native"] = (means.str.contains(r"native|indigenous")
                      & ~sinas["is_alien"])
print(f"   {len(sinas):,} rows | {sinas['taxon_clean'].nunique():,} names "
      f"| {sinas['location'].nunique()} locations | kingdom hint: "
      f"{kingdom_col or 'none'}")
print(means.value_counts().head(10).to_string())

Step 1: SInAS
   SInAS_3.1.1.csv: encoding=utf-8-sig, sep=' '
   427,956 rows | 41,171 names | 289 locations | kingdom hint: none
establishmentMeans
native                   240855
introduced               174034
introduced; uncertain      8182
uncertain                  4879
vagrant                       5
uncertain; introduced         1


## Step 2 · Match SInAS names to the GBIF Backbone

Every distinct name is sent to GBIF's matcher once (answers are cached).
`resolve_matches` then decides what each name means in the graph:

| `match_flag` | Meaning | Used as species? |
|---|---|---|
| `exact` | name found as written | yes |
| `fuzzy` | small spelling difference, confidence ≥ `MIN_FUZZY_CONFIDENCE` | yes |
| `fuzzy_low_confidence` | doubtful spelling match | **no** |
| `higherrank` | only the genus (or higher) was found | **no** |
| `no_match` | nothing found | **no** |

In [8]:
print("Step 2: SInAS -> GBIF Backbone")
sinas_pairs = (sinas[["taxon_clean", "kingdom_hint"]].dropna(
    subset=["taxon_clean"]).drop_duplicates())
sinas_pairs = list(sinas_pairs.itertuples(index=False, name=None))
sinas_res = resolve_matches(match_names(sinas_pairs, "SInAS names"),
                            allow_higherrank=False)
print(sinas_res["match_flag"].value_counts().to_string())
print(f"   species-level keys: {sinas_res['species_key'].nunique():,}")

Step 2: SInAS -> GBIF Backbone
   SInAS names: 41,171 requests | 0 cached | 41,171 to query


SInAS names:   0%|          | 0/9 [00:00<?, ?batch/s]

match_flag
exact                   38609
higherrank               1407
fuzzy                     830
no_match                  302
fuzzy_low_confidence       23
   species-level keys: 34,121


## Step 3 · GloBI pass 1 — collect partner names

The GloBI file is streamed in chunks (it does not fit in memory as a
whole). For every source and target we keep the name, the kingdom GloBI
reports (used as a matching hint) and any GBIF ID GloBI already attached
(used as a fallback in Step 4).

In [9]:
print("Step 3: GloBI pass 1 (names)")
GLOBI_READ = dict(sep="\t", dtype=str, chunksize=GLOBI_CHUNK,
                  quoting=csv.QUOTE_NONE, on_bad_lines="skip",
                  compression="infer")
SIDE_COLS = {side: [f"{side}TaxonName", f"{side}TaxonKingdomName",
                    f"{side}TaxonIds"] for side in ("source", "target")}
pass1_cols = SIDE_COLS["source"] + SIDE_COLS["target"] + [
    "interactionTypeName"]

name_counts = Counter()
gbif_id_hint = {}
term_counts = Counter()
n_rows = 0
for chunk in tqdm(pd.read_csv(GLOBI_TSV, usecols=pass1_cols, **GLOBI_READ),
                  desc="GloBI pass 1", unit="chunk"):
    n_rows += len(chunk)
    term_counts.update(chunk["interactionTypeName"].dropna())
    for side, (c_name, c_king, c_ids) in SIDE_COLS.items():
        names = chunk[c_name].map(clean_name)
        kings = chunk[c_king].map(clean_kingdom)
        ok = names.notna()
        name_counts.update(zip(names[ok], kings[ok]))
        ids = chunk.loc[ok, c_ids].str.extract(r"GBIF:(\d+)",
                                               expand=False)
        for key, gid in zip(zip(names[ok][ids.notna()],
                                kings[ok][ids.notna()]),
                            ids.dropna().astype(int)):
            gbif_id_hint.setdefault(key, gid)

globi_pairs = list(name_counts)
print(f"   {n_rows:,} GloBI records | {len(globi_pairs):,} distinct "
      f"(name, kingdom) | {len(gbif_id_hint):,} with a GloBI GBIF ID")

Step 3: GloBI pass 1 (names)


GloBI pass 1: 0chunk [00:00, ?chunk/s]

   20,361,182 GloBI records | 1,087,885 distinct (name, kingdom) | 393,703 with a GloBI GBIF ID


## Step 4 · Match GloBI names to the GBIF Backbone

Same matcher and rules as for SInAS, except that a GloBI name matching
only a genus (or family / order / class) is kept as a **higher-order
endpoint** (`ACCEPT_HIGHERRANK_FOR_GLOBI`). Names that GBIF cannot match
but that GloBI already linked to a GBIF ID are looked up by that ID.

In [10]:
print("Step 4: GloBI -> GBIF Backbone")
globi_res = resolve_matches(match_names(globi_pairs, "GloBI names"),
                            allow_higherrank=ACCEPT_HIGHERRANK_FOR_GLOBI)

# --- fallback: GloBI's own GBIF IDs for names the matcher could not place
unplaced = set(zip(globi_res.loc[globi_res["node_key"].isna(), "name"],
                   globi_res.loc[globi_res["node_key"].isna(),
                                 "kingdom_hint"]))
fallback_ids = {key: gid for key, gid in gbif_id_hint.items()
                if key in unplaced}
if fallback_ids:
    looked_up = lookup_usage_keys(sorted(set(fallback_ids.values())),
                                  "GloBI GBIF IDs")
    rows = []
    for (name, kingdom), gid in fallback_ids.items():
        rec = looked_up.get(gid)
        if rec and rec.get("matchType") == "GLOBI_ID":
            rows.append({"name": name, "kingdom_hint": kingdom,
                         **{f: rec.get(f) for f in MATCH_FIELDS}})
    if rows:
        fb = resolve_matches(pd.DataFrame(rows), allow_higherrank=True)
        fb = fb[fb["node_key"].notna()]
        globi_res = pd.concat([
            globi_res.set_index(["name", "kingdom_hint"]).drop(
                fb.set_index(["name", "kingdom_hint"]).index,
                errors="ignore").reset_index(),
            fb], ignore_index=True)
globi_res["kingdom_hint"] = globi_res["kingdom_hint"].map(clean_kingdom)
globi_res["n_records"] = [name_counts[(n, k)] for n, k in
                          zip(globi_res["name"], globi_res["kingdom_hint"])]
print(globi_res["match_flag"].value_counts().to_string())
print("   node ranks:",
      globi_res["node_rank"].value_counts(dropna=False).to_dict())

Step 4: GloBI -> GBIF Backbone
   GloBI names: 1,087,885 requests | 7,492 cached | 1,080,393 to query


GloBI names:   0%|          | 0/217 [00:00<?, ?batch/s]

   GloBI GBIF IDs: 2,207 requests | 0 cached | 2,207 to query


GloBI GBIF IDs:   0%|          | 0/1 [00:00<?, ?batch/s]

C:\Users\simon\AppData\Local\Temp\ipykernel_28600\4126819709.py:23: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  globi_res = pd.concat([


match_flag
no_match                498610
exact                   451134
higherrank               96709
fuzzy                    26893
fuzzy_low_confidence     12012
globi_gbif_id             2527
   node ranks: {None: 578153, 'SPECIES': 418944, 'GENUS': 83164, 'FAMILY': 6087, 'ORDER': 945, 'CLASS': 592}


## Step 5 · Interaction ontology

Every raw GloBI interaction term is classified once with `classify_term`.
The table shows each term, its category, the model edge type, the valence
for source and target, how it was classified, and how many records use it.

**Please check the `heuristic` and `unmatched` rows**: they are guesses or
unknowns. Add any important term to `TERM_TABLE` in the helper cell and
re-run from there.

In [13]:
print("Step 5: interaction ontology")
ontology = pd.DataFrame(
    [(term, n, *classify_term(term)) for term, n in term_counts.items()],
    columns=["interaction_type", "n_records", "category", "gnn_category",
             "source_valence", "target_valence", "rule"],
).sort_values("n_records", ascending=False).reset_index(drop=True)
share = ontology.groupby("rule")["n_records"].sum() / \
    ontology["n_records"].sum()
print("   share of records by rule:", share.round(3).to_dict())
print("   share of records by category:",
      (ontology.groupby("category")["n_records"].sum()
       / ontology["n_records"].sum()).round(3).to_dict())
ontology[ontology["rule"] != "table"].head(25)

Step 5: interaction ontology
   share of records by rule: {'table': 1.0, 'unmatched': 0.0}
   share of records by category: {'antagonism': 0.001, 'commensal': 0.001, 'commensal_host': 0.0, 'enemy': 0.0, 'mutualism': 0.064, 'neutral': 0.193, 'resource': 0.741, 'unknown': 0.0}


,interaction_type,n_records,category,gnn_category,source_valence,target_valence,rule
42,farms,1,unknown,none,0,0,unmatched


## Step 6 · GloBI pass 2 — extract the network around SInAS

Every GloBI record is mapped to graph nodes. A record is **kept** when

* one side is a SInAS species, **or**
* one side is a higher taxon (`HOI_RANKS`) that is an ancestor
  (`HOI_ANCESTOR_LEVELS`) of at least one SInAS species, e.g. a genus that
  contains a SInAS species,

and **both** sides could be placed in the backbone (a partner we cannot
identify carries no usable information). Self-loops (a taxon interacting
with itself) are dropped.

Identical (source, target, interaction type) records are merged;
`n_records` and `n_references` keep the amount of evidence, which can be
used as an edge weight.

In [14]:
print("Step 6: GloBI pass 2 (edges)")
focal_species = set(sinas_res["species_key"].dropna().astype(int))
focal_ancestors = set()
sp_rows = sinas_res[sinas_res["species_key"].notna()]
for level in HOI_ANCESTOR_LEVELS:
    focal_ancestors |= set(pd.to_numeric(sp_rows[f"{level}Key"],
                                         errors="coerce").dropna()
                           .astype(int))

node_of = {(n, k): (int(key), rank) for n, k, key, rank in zip(
    globi_res["name"], globi_res["kingdom_hint"], globi_res["node_key"],
    globi_res["node_rank"]) if pd.notna(key)}

pass2_cols = (SIDE_COLS["source"][:2] + SIDE_COLS["target"][:2]
              + ["interactionTypeName", "referenceCitation"])
parts = []
dropped = Counter()
for chunk in tqdm(pd.read_csv(GLOBI_TSV, usecols=pass2_cols, **GLOBI_READ),
                  desc="GloBI pass 2", unit="chunk"):
    cols = {}
    for side in ("source", "target"):
        keys = list(zip(chunk[f"{side}TaxonName"].map(clean_name),
                        chunk[f"{side}TaxonKingdomName"].map(clean_kingdom)))
        mapped = [node_of.get(k, (None, None)) for k in keys]
        cols[f"{side}_key"] = pd.array([m[0] for m in mapped],
                                       dtype="Int64")
        cols[f"{side}_rank"] = [m[1] for m in mapped]
    df = pd.DataFrame(cols, index=chunk.index)
    df["interaction_type"] = chunk["interactionTypeName"].str.strip()
    df["reference"] = chunk["referenceCitation"]

    s_sp = df["source_rank"].eq("SPECIES")
    t_sp = df["target_rank"].eq("SPECIES")
    relevant = ((s_sp & df["source_key"].isin(focal_species))
                | (t_sp & df["target_key"].isin(focal_species))
                | (~s_sp & df["source_key"].isin(focal_ancestors))
                | (~t_sp & df["target_key"].isin(focal_ancestors)))
    both = df["source_key"].notna() & df["target_key"].notna()
    self_loop = (df["source_key"] == df["target_key"]).fillna(False)
    self_loop = self_loop.astype(bool)
    dropped["one_side_unplaced"] += int((relevant & ~both).sum())
    dropped["self_loop"] += int((relevant & both & self_loop).sum())
    parts.append(df[relevant & both & ~self_loop])

raw_edges = pd.concat(parts, ignore_index=True)
del parts
edges = (raw_edges.groupby(["source_key", "target_key",
                            "interaction_type"], dropna=False)
         .agg(source_rank=("source_rank", "first"),
              target_rank=("target_rank", "first"),
              n_records=("reference", "size"),
              n_references=("reference", "nunique"),
              reference=("reference", "first"))
         .reset_index())
term_info = {t: classify_term(t)
             for t in edges["interaction_type"].dropna().unique()}
cls = pd.DataFrame(
    [term_info.get(t, classify_term(None))
     for t in edges["interaction_type"]],
    columns=["category", "gnn_category", "source_valence",
             "target_valence", "ontology_rule"], index=edges.index)
edges = pd.concat([edges, cls], axis=1)
print(f"   relevant records: {len(raw_edges):,} -> unique edges: "
      f"{len(edges):,} | dropped: {dict(dropped)}")
print(edges.groupby(["source_rank", "target_rank"]).size()
      .sort_values(ascending=False).head(12).to_string())

Step 6: GloBI pass 2 (edges)


GloBI pass 2: 0chunk [00:00, ?chunk/s]

   relevant records: 10,610,127 -> unique edges: 1,710,608 | dropped: {'one_side_unplaced': 1044460, 'self_loop': 42455}
source_rank  target_rank
SPECIES      SPECIES        991958
             GENUS          327369
GENUS        SPECIES        148250
SPECIES      FAMILY          66379
GENUS        GENUS           60423
SPECIES      CLASS           29121
             ORDER           23688
FAMILY       SPECIES         19169
GENUS        FAMILY          10003
FAMILY       GENUS            7161
ORDER        SPECIES          6921
CLASS        SPECIES          4402


## Step 7 · Taxonomy backbone table

One row per taxon used anywhere in the graph (SInAS species, GloBI
species partners and higher-order endpoints), with its rank, accepted name
and all parent keys and names. The model can use this to give **every**
species node taxonomy edges, including GloBI partners that are not in
SInAS (previously only SInAS species had them).

In [15]:
print("Step 7: taxonomy table")
NAME_BY_RANK = {"SPECIES": "species", "GENUS": "genus", "FAMILY": "family",
                "ORDER": "order", "CLASS": "class"}
all_res = pd.concat([sinas_res.assign(_src="sinas"),
                     globi_res.assign(_src="globi")], ignore_index=True)
all_res = all_res[all_res["node_key"].notna()].copy()
all_res["canonical_name"] = [
    rec.get(NAME_BY_RANK.get(rank, ""), None) or rec.get("canonicalName")
    for rec, rank in zip(all_res.to_dict("records"), all_res["node_rank"])]
tax_cols = {f"{lv}Key": f"{lv}_id" for lv in LEVELS}
tax_cols.update({lv: lv for lv in LEVELS})
taxonomy = (all_res.sort_values("match_flag")  # exact before fuzzy
            .groupby("node_key", as_index=False)
            .agg(rank=("node_rank", "first"),
                 canonical_name=("canonical_name", "first"),
                 **{v: (k, "first") for k, v in tax_cols.items()}))
taxonomy = taxonomy.rename(columns={"node_key": "gbif_id"})
for lv in LEVELS:
    taxonomy[f"{lv}_id"] = pd.to_numeric(taxonomy[f"{lv}_id"],
                                         errors="coerce").astype("Int64")
endpoints = set(edges["source_key"].dropna()) | set(
    edges["target_key"].dropna())
taxonomy["in_sinas"] = taxonomy["gbif_id"].isin(focal_species)
taxonomy["is_globi_endpoint"] = taxonomy["gbif_id"].isin(endpoints)
taxonomy = taxonomy[taxonomy["in_sinas"] | taxonomy["is_globi_endpoint"]]
print(f"   {len(taxonomy):,} taxa:",
      taxonomy["rank"].value_counts().to_dict())

Step 7: taxonomy table
   241,834 taxa: {'SPECIES': 215744, 'GENUS': 23141, 'FAMILY': 2464, 'ORDER': 386, 'CLASS': 99}


## Step 8 · Assemble the SInAS output

SInAS rows are joined to their match. Only rows with a **species-level**
key are written to `sinas_matched_species.csv`; the others stay in the
audit file. Columns keep the names the model notebook expects
(`gbif_id`, `gbif_canonical_name`, `gbif_genus_id`, …) and add:

| Column | Meaning |
|---|---|
| `gbif_usage_key` | key of the exact name matched (may be a synonym or subspecies) |
| `gbif_status` | ACCEPTED / SYNONYM / DOUBTFUL |
| `match_flag` | see Step 2 |
| `has_globi_interactions` | species is an endpoint of ≥ 1 species-level edge |
| `has_globi_hoi` | species' genus / family / … is an endpoint of ≥ 1 HOI edge |
| `n_globi_partners` | number of distinct species-level partners |

In [16]:
print("Step 8: SInAS output")
res_cols = (["name", "kingdom_hint", "species_key", "usageKey", "status",
             "matchType", "confidence", "rank", "match_flag", "species"]
            + [f"{lv}Key" for lv in LEVELS])
sinas_out = sinas.merge(
    sinas_res[res_cols].rename(columns={"name": "taxon_clean"}),
    on=["taxon_clean", "kingdom_hint"], how="left")
sinas_out = sinas_out.rename(columns={
    "species_key": "gbif_id", "usageKey": "gbif_usage_key",
    "species": "gbif_canonical_name", "status": "gbif_status",
    "matchType": "gbif_match_type", "confidence": "gbif_confidence",
    "rank": "gbif_rank",
    **{f"{lv}Key": f"gbif_{lv}_id" for lv in LEVELS}})

sp_edges = edges[edges["source_rank"].eq("SPECIES")
                 & edges["target_rank"].eq("SPECIES")]
partners = pd.concat([
    sp_edges[["source_key", "target_key"]].set_axis(["sp", "partner"],
                                                    axis=1),
    sp_edges[["target_key", "source_key"]].set_axis(["sp", "partner"],
                                                    axis=1)])
n_partners = partners.drop_duplicates().groupby("sp").size()
hoi_nodes = set(edges.loc[~edges["source_rank"].eq("SPECIES"),
                          "source_key"]) | set(
    edges.loc[~edges["target_rank"].eq("SPECIES"), "target_key"])

sinas_out["n_globi_partners"] = (sinas_out["gbif_id"].map(n_partners)
                                 .fillna(0).astype(int))
sinas_out["has_globi_interactions"] = sinas_out["n_globi_partners"] > 0
anc = sinas_out[[f"gbif_{lv}_id" for lv in HOI_ANCESTOR_LEVELS]]
sinas_out["has_globi_hoi"] = anc.isin(hoi_nodes).any(axis=1)

matched_mask = sinas_out["gbif_id"].notna()
unmatched_rows = sinas_out[~matched_mask]
sinas_out = sinas_out[matched_mask].copy()
sinas_out["gbif_id"] = sinas_out["gbif_id"].astype("int64")
print(f"   kept {len(sinas_out):,} of {len(sinas):,} rows "
      f"({sinas_out['gbif_id'].nunique():,} species); "
      f"{len(unmatched_rows):,} rows without a species-level match")
per_sp = sinas_out.groupby("gbif_id")[["has_globi_interactions",
                                        "has_globi_hoi"]].first()
print(f"   species with species-level interactions: "
      f"{per_sp['has_globi_interactions'].mean():.1%} | with HOI: "
      f"{per_sp['has_globi_hoi'].mean():.1%}")

Step 8: SInAS output
   kept 421,990 of 427,956 rows (34,121 species); 5,966 rows without a species-level match
   species with species-level interactions: 68.7% | with HOI: 99.5%


## Step 9 · Quality assurance

* match statistics for SInAS and GloBI names,
* coverage: share of SInAS species with interactions (species level / HOI),
* ontology: share of records that are neutral / unknown,
* **comparison with the previous outputs** (if they exist in `data/`):
  how many species and edges are shared, new or gone. Big differences
  are expected where this notebook fixes things (rolled-up subspecies,
  synonyms, rejected higher-rank matches) and should be inspected
  elsewhere.

In [17]:
print("Step 9: QA")
sp_species = sinas_out["gbif_id"].unique()
qa = {
    "sinas_rows": int(len(sinas)),
    "sinas_rows_kept": int(len(sinas_out)),
    "sinas_species": int(len(sp_species)),
    "sinas_match_flags": sinas_res["match_flag"].value_counts().to_dict(),
    "globi_records": int(n_rows),
    "globi_names": int(len(globi_res)),
    "globi_match_flags": globi_res["match_flag"].value_counts().to_dict(),
    "globi_names_placed_share": float(globi_res["node_key"].notna().mean()),
    "edges_total": int(len(edges)),
    "edges_species_level": int(len(sp_edges)),
    "edges_higher_order": int(len(edges) - len(sp_edges)),
    "edges_dropped": dict(dropped),
    "coverage_species_level": float(
        sinas_out.groupby("gbif_id")["has_globi_interactions"].first()
        .mean()),
    "coverage_with_hoi": float(
        (sinas_out.groupby("gbif_id")[["has_globi_interactions",
                                       "has_globi_hoi"]].first()
         .any(axis=1)).mean()),
    "edge_share_by_category": (edges.groupby("category")["n_records"].sum()
                               / edges["n_records"].sum()).round(4)
    .to_dict(),
    "generic_association_as": GENERIC_ASSOCIATION_AS,
}

# --- comparison with the previous run ----------------------------------------
previous = {}
for path, key in ((SINAS_OUT, "gbif_id"), (NET_SPECIES_OUT, None)):
    if path.exists():
        old = pd.read_csv(path, low_memory=False)
        if key:
            old_ids = set(pd.to_numeric(old[key], errors="coerce")
                          .dropna().astype(int))
            new_ids = set(sp_species)
            previous["species"] = {
                "previous": len(old_ids), "now": len(new_ids),
                "shared": len(old_ids & new_ids),
                "only_previous": len(old_ids - new_ids),
                "only_now": len(new_ids - old_ids)}
        else:
            def _pairs(d, s, t):
                d = d.dropna(subset=[s, t])
                return set(zip(d[s].astype(int), d[t].astype(int)))
            old_p = _pairs(old, "source_gbif_id", "target_gbif_id")
            new_p = _pairs(sp_edges, "source_key", "target_key")
            previous["species_edges"] = {
                "previous": len(old_p), "now": len(new_p),
                "shared": len(old_p & new_p)}
qa["comparison_with_previous"] = previous
print(json.dumps({k: v for k, v in qa.items()
                  if not isinstance(v, dict)}, indent=1))
print(json.dumps(previous, indent=1))

Step 9: QA
{
 "sinas_rows": 427956,
 "sinas_rows_kept": 421990,
 "sinas_species": 34121,
 "globi_records": 20361182,
 "globi_names": 1087885,
 "globi_names_placed_share": 0.46855320185497545,
 "edges_total": 1710608,
 "edges_species_level": 991958,
 "edges_higher_order": 718650,
 "coverage_species_level": 0.6866445883766595,
 "coverage_with_hoi": 0.9970985610034876,
 "generic_association_as": "neutral"
}
{
 "species": {
  "previous": 26541,
  "now": 34121,
  "shared": 24722,
  "only_previous": 1819,
  "only_now": 9399
 },
 "species_edges": {
  "previous": 807511,
  "now": 836853,
  "shared": 759503
 }
}


## Step 10 · Write outputs

If `BACKUP_PREVIOUS` is on, existing outputs are first renamed to
`*_previous.csv` so the old and new versions can be compared later.

In [18]:
print("Step 10: writing outputs")


def _backup(path):
    """Rename an existing output to *_previous.* before overwriting it."""
    if BACKUP_PREVIOUS and path.exists():
        path.replace(path.with_name(path.stem + "_previous" + path.suffix))


name_of = dict(zip(taxonomy["gbif_id"], taxonomy["canonical_name"]))
net = edges.rename(columns={"source_key": "source_gbif_id",
                            "target_key": "target_gbif_id"})
net.insert(0, "source_taxon_name", net["source_gbif_id"].map(name_of))
net.insert(4, "target_taxon_name", net["target_gbif_id"].map(name_of))
is_sp = net["source_rank"].eq("SPECIES") & net["target_rank"].eq("SPECIES")

audit = pd.concat([
    sinas_res.assign(dataset="SInAS"),
    globi_res.assign(dataset="GloBI")], ignore_index=True)[
    ["dataset", "name", "kingdom_hint", "match_flag", "matchType",
     "confidence", "rank", "status", "usageKey", "species_key", "node_key",
     "node_rank", "canonicalName", "kingdom", "family", "genus"]
    + (["n_records"] if "n_records" in globi_res else [])]

outputs = [(sinas_out.drop(columns=["taxon_clean"]), SINAS_OUT),
           (taxonomy, TAXONOMY_OUT),
           (net[is_sp], NET_SPECIES_OUT),
           (net[~is_sp], NET_HIGHER_OUT),
           (ontology, ONTOLOGY_OUT),
           (audit, AUDIT_OUT)]
for frame, path in outputs:
    _backup(path)
    frame.to_csv(path, index=False)
    print(f"   {path.name}: {len(frame):,} rows")
with open(QA_OUT, "w") as fh:
    json.dump(qa, fh, indent=2, default=str)
print(f"   {QA_OUT.name}\nDONE")

Step 10: writing outputs
   sinas_matched_species.csv: 421,990 rows
   taxonomy_backbone.csv: 241,834 rows
   matched_globi_network.csv: 991,958 rows
   matched_globi_network_higher_order.csv: 718,650 rows
   interaction_type_ontology.csv: 43 rows
   name_matching_audit.csv: 1,129,056 rows
   sinas_globi_qa.json
DONE
